# Predictive Maintenance — EDA

Exploratory analysis of the generated IoT sensor dataset.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

sns.set_theme(style='darkgrid')
pd.set_option('display.max_columns', 40)

In [ ]:
# Load dataset (generate first if needed)
data_path = Path('../data/combined.csv')
if not data_path.exists():
    import subprocess, sys
    subprocess.run([sys.executable, '../data/generate_dataset.py'])

df = pd.read_csv(data_path, parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Fault label distribution
fig = px.histogram(df, x='fault_name', color='domain', barmode='group',
                   title='Fault Label Distribution by Domain')
fig.show()

In [ ]:
# Sensor correlation heatmap
sensor_cols = ['vibration','temperature','current','pressure','rpm','humidity','power_w','degradation']
corr = df[sensor_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Sensor Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Sensor distributions by fault label (industrial)
ind = df[df['domain']=='industrial']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, ['vibration','temperature','current','pressure','rpm','power_w']):
    for label in ind['fault_name'].unique():
        vals = ind[ind['fault_name']==label][col]
        ax.hist(vals, bins=40, alpha=0.5, label=label, density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle('Industrial sensor distributions by fault type')
plt.tight_layout()
plt.show()

In [ ]:
# Degradation over time for one device
device = df[df['device_id']=='motor_01'].head(500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=device['timestamp'], y=device['vibration'], name='Vibration'))
fig.add_trace(go.Scatter(x=device['timestamp'], y=device['temperature']/100, name='Temp/100'))
fig.add_trace(go.Scatter(x=device['timestamp'], y=device['degradation'], name='Degradation', 
                          line=dict(color='red', dash='dash')))
fig.update_layout(title='motor_01: sensor signals vs degradation curve')
fig.show()

In [ ]:
# Feature engineering preview
from utils.preprocessing import run_feature_pipeline
df_eng, feature_cols, _ = run_feature_pipeline(df.head(2000), fit=False)
print(f'Feature count: {len(feature_cols)}')
df_eng[feature_cols[:8]].describe()